# 2-4. 계절 효과를 통제한 날씨 민감도 재분석 (편상관)

2-3차에서 계산한 상관관계에 계절 혼입(confounding) 가능성이 확인됐다
(공공기관, 제조/도매처럼 날씨와 무관해 보이는 카테고리도 기온과 상관관계가 나옴 —
날씨 자체가 아니라 계절성이 같이 움직였을 가능성).

방법: 소분류x권역x월 조합의 평균값을 빼서(그룹 평균 제거) '그 달 평균 대비 편차'만 남긴 뒤,
그 편차끼리 상관관계를 다시 계산한다. 이러면 "겨울이라서"가 아니라
"그 겨울 안에서도 유난히 추웠던 날" 같은 순수 변동만 남는다.

2-3차 결과와 비교해서, 상관관계가 계절 통제 후 사라지는 카테고리(계절 혼입이었던 것)와
여전히 남아있는 카테고리(진짜 날씨 효과)를 구분한다.

In [ ]:
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

system = platform.system()
if system == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif system == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False

pd.set_option("display.max_columns", None)
%matplotlib inline

## 2-4-0. 데이터 로드 및 집계

In [ ]:
from pathlib import Path

CATEGORY_DIR = Path("../../data/processed/consume_weather_by_category")
GROUP_ORDER = ["필수", "애매", "불필요"]

group_dfs = {}
for group in GROUP_ORDER:
    g_df = pd.read_parquet(CATEGORY_DIR / f"{group}.parquet")
    g_df["분류등급"] = group
    group_dfs[group] = g_df

df = pd.concat(group_dfs.values(), ignore_index=True)
df["date"] = pd.to_datetime(df["date"])

agg = (
    df.groupby(["date", "지점명", "소분류", "분류등급"])
    .agg(매출금액=("매출금액", "sum"), 평균기온=("평균기온(°C)", "first"), 강수량=("일강수량(mm)", "first"))
    .reset_index()
)
agg["log_매출금액"] = np.log1p(agg["매출금액"])
agg["월"] = agg["date"].dt.month

print("집계 후 shape:", agg.shape)
agg.head()

## 2-4-1. 소분류x권역x월 그룹 평균 제거 (잔차 계산)

각 소분류 안에서, (권역, 월) 조합별 평균을 빼서 잔차(그 달 그 지역 평균 대비 편차)를 만든다.

In [ ]:
agg["기온잔차"] = agg["평균기온"] - agg.groupby(["소분류", "지점명", "월"])["평균기온"].transform("mean")
agg["강수잔차"] = agg["강수량"] - agg.groupby(["소분류", "지점명", "월"])["강수량"].transform("mean")
agg["매출잔차"] = agg["log_매출금액"] - agg.groupby(["소분류", "지점명", "월"])["log_매출금액"].transform("mean")

agg[["소분류", "지점명", "월", "평균기온", "기온잔차", "log_매출금액", "매출잔차"]].head()

## 2-4-2. 소분류별 편상관계수 계산 (계절 통제 후)

In [ ]:
results = []
for (category, group), sub in agg.groupby(["소분류", "분류등급"]):
    if len(sub) < 20:
        continue

    temp_corr, temp_p = stats.spearmanr(sub["기온잔차"], sub["매출잔차"])
    rain_corr, rain_p = stats.spearmanr(sub["강수잔차"], sub["매출잔차"])

    results.append({
        "소분류": category,
        "분류등급": group,
        "표본수": len(sub),
        "기온편상관": temp_corr,
        "기온_p값": temp_p,
        "강수편상관": rain_corr,
        "강수_p값": rain_p,
    })

partial_df = pd.DataFrame(results).sort_values("기온편상관", ascending=False)
print(f"총 {len(partial_df)}개 소분류 분석됨")
partial_df

## 2-4-3. 계절 통제 전후 비교

2-3차(원본 상관계수)와 2-4차(편상관계수)를 나란히 비교해서, 계절 통제 후
상관관계가 크게 줄어든(계절 혼입이었던) 카테고리와 여전히 남아있는(진짜 날씨 효과) 카테고리를 구분한다.

주의: 이 셀은 2-3차EDA.ipynb를 먼저 실행해서 원본 상관계수를 다시 계산해야 하므로,
여기서는 2-3차와 동일한 방식으로 원본(비통제) 상관계수를 다시 계산한다.

In [ ]:
raw_results = []
for (category, group), sub in agg.groupby(["소분류", "분류등급"]):
    if len(sub) < 20:
        continue
    temp_corr, _ = stats.spearmanr(sub["평균기온"], sub["log_매출금액"])
    raw_results.append({"소분류": category, "원본_기온상관": temp_corr})

raw_df = pd.DataFrame(raw_results)

compare_df = partial_df.merge(raw_df, on="소분류")
compare_df["감소폭"] = compare_df["원본_기온상관"].abs() - compare_df["기온편상관"].abs()
compare_df = compare_df.sort_values("감소폭", ascending=False)

print("=== 계절 통제 후 상관관계가 가장 많이 줄어든(=계절 혼입이었을 가능성 높은) 상위 10개 ===")
print(compare_df[["소분류", "분류등급", "원본_기온상관", "기온편상관", "감소폭"]].head(10))

print()
print("=== 계절 통제 후에도 상관관계가 유지/증가한(=진짜 날씨 효과일 가능성 높은) 상위 10개 ===")
print(compare_df[["소분류", "분류등급", "원본_기온상관", "기온편상관", "감소폭"]].tail(10))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
colors = {"필수": "tomato", "애매": "goldenrod", "불필요": "gray"}
for g in GROUP_ORDER:
    sub = compare_df[compare_df["분류등급"] == g]
    ax.scatter(sub["원본_기온상관"], sub["기온편상관"], label=g, alpha=0.7, color=colors[g])

lims = [-0.3, 0.3]
ax.plot(lims, lims, "k--", linewidth=0.8, label="변화 없음 (대각선)")
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("원본 기온상관계수 (계절 미통제)")
ax.set_ylabel("기온편상관계수 (계절 통제)")
ax.set_title("계절 통제 전후 기온상관계수 비교\n(대각선에 가까울수록 계절 혼입 아님, 원점 쪽으로 이동할수록 계절 혼입이었음)")
ax.legend()
plt.tight_layout()
plt.show()

## 2-4-4. 재분류 및 대조군(불필요) 검증

계절 통제 후에도 '불필요' 그룹이 여전히 날씨무관 쪽에 많이 몰려있는지 다시 확인한다.

In [ ]:
def classify(row):
    if row["기온편상관"] > 0.15:
        return "고온민감"
    elif row["기온편상관"] < -0.15:
        return "저온민감"
    elif abs(row["강수편상관"]) > 0.1:
        return "강수민감"
    else:
        return "날씨무관"

partial_df["날씨민감도_재분류"] = partial_df.apply(classify, axis=1)

print(partial_df["날씨민감도_재분류"].value_counts())
print()
print("=== 분류등급 x 날씨민감도(계절통제 후) 교차표 ===")
print(pd.crosstab(partial_df["분류등급"], partial_df["날씨민감도_재분류"]))

## 2-4 요약 (직접 채워넣기)

- 계절 혼입으로 확인된 카테고리 (원본 상관 강했으나 통제 후 약해짐): ...
- 진짜 날씨 효과로 확인된 카테고리 (통제 후에도 상관관계 유지): ...
- 불필요 그룹 대조군 검증: 계절 통제 후 [개선됨 / 여전히 애매함] ...

다음 단계: 진짜 날씨 효과가 확인된 카테고리만 추려서 회귀분석/심화 모델링 진행.